# grid-rbd JAX backend — jit, vmap & differentiation

The **JAX backend** wraps the same compiled per-robot `.so` behind
[`jax.ffi`](https://docs.jax.dev/en/latest/ffi.html) targets, so GRiD's
dynamics/kinematics/gradient kernels are callable inside `jax.jit` and run
device-resident on JAX-managed CUDA streams. Register with
`grid_rbd.jax.register_robot(...)` to get a `JaxRobotHandle` whose methods
return `jax.Array`s.

This is the JAX parallel to notebook **02** (torch). We show the JAX-specific
value — **`jax.jit`** and **`jax.vmap`** — plus how differentiation works on
this backend (see section 4: the FFI calls are not auto-differentiable, so the
derivatives come from GRiD's own **analytic** gradient kernels, which we
cross-check by finite difference).

**Setup:** a CUDA GPU + `nvcc` on PATH, and the JAX extra installed editable
from this repo — `pip install -e ".[jax]"` from the repo root (see
[`notebooks/README.md`](README.md)). iiwa14 caches in seconds; the first
`jax.jit` trace adds a little XLA warm-up.

In [1]:
import numpy as np
import jax, jax.numpy as jnp
from pathlib import Path
import grid_rbd            # numpy handle, for the cross-check oracle
import grid_rbd.jax as grid_jax

URDF = next(p / 'config/robot_assets' / 'iiwa14.urdf' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'config/robot_assets' / 'iiwa14.urdf').exists())
assert URDF.exists(), URDF
np.random.seed(0)
print('jax', jax.__version__, ' devices:', jax.devices())

jax 0.10.0  devices: [CudaDevice(id=0)]


## 1. Register with the JAX backend

`grid_jax.register_robot` compiles/caches the **same** `.so` as the plain
`grid_rbd.register_robot` (cache hit if already built) and additionally
registers the JAX FFI targets. We also grab a plain numpy handle on the same
robot to use as a validation oracle.

In [2]:
h = grid_jax.register_robot('iiwa14_jax_nb', urdf_path=str(URDF),
                            floating_base=False, max_batch_size=64)
h_np = grid_rbd.register_robot('iiwa14_jax_nb_oracle', urdf_path=str(URDF),
                               floating_base=False, max_batch_size=64)
NJ, NV = h.num_joints, h.num_vel
print(h.__class__.__name__, ' NJ =', NJ, ' NV =', NV)

JaxRobotHandle  NJ = 7  NV = 7


## 2. `jax.jit` a dynamics call

The handle methods are `jax.jit`-traceable: the FFI target executes inside the
compiled XLA program. We jit `forward_dynamics` and validate against the numpy
handle (same `.so`, so they agree to float32 round-off).

In [3]:
B = 8
q  = jnp.asarray(np.random.randn(B, NJ), dtype=jnp.float32)
qd = jnp.asarray(np.random.randn(B, NJ), dtype=jnp.float32)
u  = jnp.asarray(np.random.randn(B, NJ), dtype=jnp.float32)

@jax.jit
def fd(q, qd, u):
    return h.forward_dynamics(q, qd, u)

qdd = fd(q, qd, u)                 # traced + compiled once, then cached
qdd_np = h_np.forward_dynamics(np.asarray(q), np.asarray(qd), np.asarray(u))
err = float(jnp.max(jnp.abs(qdd - jnp.asarray(qdd_np))))
print('jit forward_dynamics:', qdd.shape, '  max|jax - numpy| =', f'{err:.2e}')
assert qdd.shape == (B, NJ)
assert err < 1e-4, err

jit forward_dynamics: (8, 7)   max|jax - numpy| = 0.00e+00


## 3. `jax.vmap` over a batch

The kernels already batch over axis 0, and the core FFI calls now carry
`vmap_method="broadcast_all"`, so `jax.vmap` works directly: write the
per-sample computation and map it — JAX re-adds the mapped axis and dispatches a
**single native batched kernel** (not a Python loop). We map a single-state
`forward_dynamics` and check it matches the natively-batched call.

In [4]:
# jax.vmap maps the per-sample (NJ,) computation over the leading batch axis;
# vmap_method="broadcast_all" collapses it back into ONE native (B, NJ) kernel.
qdd_vmap = jax.vmap(h.forward_dynamics)(q, qd, u)
qdd_batched = h.forward_dynamics(q, qd, u)        # native batch over axis 0
err = float(jnp.max(jnp.abs(qdd_vmap - qdd_batched)))
print('vmap forward_dynamics:', qdd_vmap.shape, '  max|vmap - native| =', f'{err:.2e}')
assert qdd_vmap.shape == (B, NJ)
assert err == 0.0, err                            # same kernel, identical result

# vmap composes with jit too (single compiled, batched call).
err_jit = float(jnp.max(jnp.abs(jax.jit(jax.vmap(h.forward_dynamics))(q, qd, u) - qdd_batched)))
print('jit(vmap) forward_dynamics: max|err| =', f'{err_jit:.2e}'); assert err_jit == 0.0

vmap forward_dynamics: (8, 7)   max|vmap - native| = 0.00e+00
jit(vmap) forward_dynamics: max|err| = 0.00e+00


## 4. Differentiation — `jax.grad` / `jax.jacobian` via analytic kernels

GRiD emits **analytic** gradients (there is no autodiff tape), so each core
forward op is wrapped with a `jax.custom_vjp` whose backward pass contracts the
cotangent with GRiD's matching analytic-gradient kernel — itself an FFI call.
The result: `jax.grad` / `jax.jacobian` / `jax.vjp` **through**
`forward_dynamics`, `inverse_dynamics` and `end_effector_pose` now work,
are `jax.jit`-compatible, and reproduce the analytic Jacobian **exactly**:

| forward op | VJP uses |
|---|---|
| `forward_dynamics(q,qd,u)` | `forward_dynamics_gradient` (∂qdd/∂q, ∂qdd/∂qd) + `minv` (∂qdd/∂u = M⁻¹) |
| `inverse_dynamics(q,qd)` | `inverse_dynamics_gradient` (∂c/∂q, ∂c/∂qd) |
| `end_effector_pose(q)` | `end_effector_pose_gradient` (tangent-space Jacobian) |

Below we differentiate `forward_dynamics` and confirm `jax.jacobian` matches the
analytic `forward_dynamics_gradient` block-for-block. The standalone analytic
kernels (`idsva_so` / `fdsva_so`, second order) remain available too (section 5).

> Inputs to the differentiable ops must carry a leading batch axis `(B, NJ)`
> when called directly; under `jax.vmap` each per-sample `(NJ,)` slice is fine.

In [5]:
# jax.grad / jax.jacobian now flow THROUGH the FFI forward op via custom_vjp.
q1  = jnp.asarray(np.random.randn(1, NJ) * 0.3, dtype=jnp.float32)
qd1 = jnp.asarray(np.random.randn(1, NJ) * 0.3, dtype=jnp.float32)
u1  = jnp.asarray(np.random.randn(1, NJ) * 0.3, dtype=jnp.float32)

# jit(grad(...)) of a scalar loss through forward_dynamics — finite + compiles.
loss = lambda qq: jnp.sum(h.forward_dynamics(qq, qd1, u1) ** 2)
g = jax.jit(jax.grad(loss))(q1)
print('jit(grad) through forward_dynamics:', g.shape, ' finite =', bool(np.all(np.isfinite(g))))
assert g.shape == (1, NJ) and bool(np.all(np.isfinite(g)))

# jax.jacobian d(qdd)/dq must equal the analytic forward_dynamics_gradient dq block.
J = jax.jacobian(lambda qq: h.forward_dynamics(qq[None], qd1, u1)[0])(q1[0])  # (NJ, NJ)
G = np.asarray(h.forward_dynamics_gradient(q1, qd1, u1))[0]                    # (NJ, 2*NJ)
rel = float(np.max(np.abs(np.asarray(J) - G[:, :NJ]))) / max(1.0, float(np.max(np.abs(G[:, :NJ]))))
print('jax.jacobian dqdd/dq vs analytic: rel|err| =', f'{rel:.2e}')
assert rel < 5e-3, rel

jit(grad) through forward_dynamics: (1, 7)  finite = True
jax.jacobian dqdd/dq vs analytic: rel|err| = 0.00e+00


In [6]:
# The supported path: GRiD's analytic forward-dynamics gradient, jitted.
q1  = jnp.asarray(np.random.randn(1, NJ) * 0.3, dtype=jnp.float32)
qd1 = jnp.asarray(np.random.randn(1, NJ) * 0.3, dtype=jnp.float32)
u1  = jnp.asarray(np.random.randn(1, NJ) * 0.3, dtype=jnp.float32)

fd_grad = jax.jit(h.forward_dynamics_gradient)
G = fd_grad(q1, qd1, u1)               # (1, NJ, 2*NJ) = [dqdd/dq | dqdd/dqd]
df_dq, df_dqd = G[..., :NJ], G[..., NJ:]
print('forward_dynamics_gradient:', G.shape)

# Central finite-difference of forward_dynamics for the cross-check.
def fd_jac(fn, x, eps=1e-3):
    n = x.shape[1]
    J = np.zeros((NJ, n))
    for j in range(n):
        dx = np.zeros_like(x); dx[0, j] = eps
        J[:, j] = (np.asarray(fn(x + dx))[0] - np.asarray(fn(x - dx))[0]) / (2 * eps)
    return J

fdq  = fd_jac(lambda qq: h.forward_dynamics(qq, qd1, u1), q1)
fdqd = fd_jac(lambda vv: h.forward_dynamics(q1, vv, u1), qd1)
e_dq  = float(np.max(np.abs(np.asarray(df_dq[0]) - fdq)))
e_dqd = float(np.max(np.abs(np.asarray(df_dqd[0]) - fdqd)))
print(f'analytic dqdd/dq  vs FD: max|err| = {e_dq:.2e}')
print(f'analytic dqdd/dqd vs FD: max|err| = {e_dqd:.2e}')
assert e_dq < 5e-2 and e_dqd < 5e-2, (e_dq, e_dqd)

forward_dynamics_gradient: (1, 7, 14)
analytic dqdd/dq  vs FD: max|err| = 6.44e-03
analytic dqdd/dqd vs FD: max|err| = 7.58e-03


## 5. Second-order `idsva_so` vs the numpy oracle

The second-order kernel is on the JAX surface too. `idsva_so(q, qd, qdd)`
returns four `(B, NV, NV, NV)` tensors `(d2tau_dq, d2tau_dqd, d2tau_cross,
dM_dq)`; we jit it and validate block-for-block against the numpy handle
(same `.so`, so they agree to float32 round-off).

In [7]:
qdd1 = jnp.zeros((1, NJ), dtype=jnp.float32)
idsva_jax = jax.jit(h.idsva_so)(q1, qd1, qdd1)        # 4 x (1, NV, NV, NV)
idsva_np  = h_np.idsva_so(np.asarray(q1), np.asarray(qd1), np.asarray(qdd1))

names = ('d2tau_dq', 'd2tau_dqd', 'd2tau_cross', 'dM_dq')
worst = 0.0
for name, A, R in zip(names, idsva_jax, idsva_np):
    A = np.asarray(A[0]); R = np.asarray(R[0])
    scale = max(1.0, float(np.max(np.abs(R))))
    e = float(np.max(np.abs(A - R))) / scale
    print(f'  {name:11s} shape={A.shape}  max|err|/scale = {e:.2e}')
    worst = max(worst, e)
assert worst < 1e-3, worst
print('idsva_so (JAX) matches the numpy handle.')

  d2tau_dq    shape=(7, 7, 7)  max|err|/scale = 0.00e+00
  d2tau_dqd   shape=(7, 7, 7)  max|err|/scale = 0.00e+00
  d2tau_cross shape=(7, 7, 7)  max|err|/scale = 0.00e+00
  dM_dq       shape=(7, 7, 7)  max|err|/scale = 0.00e+00
idsva_so (JAX) matches the numpy handle.


JAX tour done: `jax.jit` over GRiD's dynamics, **`jax.vmap`** over the batch
axis (one native batched kernel via `vmap_method="broadcast_all"`), and
**`jax.grad` / `jax.jacobian`** through `forward_dynamics` / `inverse_dynamics`
/ `end_effector_pose` — wired by `jax.custom_vjp` to GRiD's **analytic**
gradient kernels, so the autodiff Jacobian equals the analytic one exactly and
is `jax.jit`-compatible. The standalone first/second-order kernels
(`*_gradient`, `idsva_so` / `fdsva_so`) remain on the surface too. Every number
was cross-checked, so a green *Run All* validates the JAX surface, not just
"no exception."